In [ ]:
# ============================================================
# NB_INGST_BronzeADVWDB_SalesOrderDetail
# Capa: Bronze | Origen: ADVWDB | Tabla: SalesOrderDetail
# Carga INCREMENTAL — Lee staging desde lh_landing → escribe Bronze
# FLUJO: Copy Data (SQL→lh_landing staging) → este NB → lh_bronze
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime
import re

# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
_ctx    = notebookutils.runtime.context
WS_ID   = _ctx['workspaceId']
_lh_map = {lh.displayName: lh.id for lh in notebookutils.lakehouse.list()}
ART_LANDING = _lh_map.get('lh_landing', '')
ART_BRONZE  = _lh_map.get('lh_bronze', '')



config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen       = 'ADVWDB'
      AND nombre_tabla = 'SalesOrderDetail'
      AND activo       = 1
    LIMIT 1
""").collect()[0]

nombre_tabla  = config['nombre_tabla']
origen        = config['origen']
pks           = [pk.strip() for pk in config['pks'].split(',')]
tipo_carga    = config['tipo_carga']

# Staging depositado por Copy Data en lh_landing
tabla_staging_landing = f'lh_landing.{origen}.{nombre_tabla}_staging'
# Destino Bronze
tabla_bronze  = f'lh_bronze.{origen}.{nombre_tabla}_staging'

print('✅ Config cargada')
print(f'   nombre_tabla          : {nombre_tabla}')
print(f'   origen                : {origen}')
print(f'   tipo_carga            : {tipo_carga}')
print(f'   tabla_staging_landing : {tabla_staging_landing}')
print(f'   tabla_bronze          : {tabla_bronze}')


In [ ]:
# ============================================================
# Leer staging desde lh_landing (depositado por Copy Data)
# LIMIT 1000 obligatorio (TP)
# ============================================================

try:
    df_raw = spark.sql(f'SELECT * FROM {tabla_staging_landing} LIMIT 1000')
    filas = df_raw.count()
    print(f'✅ Staging en Landing: {filas} filas')
    print(f'   Columnas: {df_raw.columns}')
    display(df_raw.limit(3))
except Exception as e:
    raise Exception(
        f'ERROR: No se encontró {tabla_staging_landing}.\n'
        f'Verificar que Copy Data del pipeline escribió en lh_landing.\n'
        f'Error: {e}'
    )

if filas == 0:
    print('ℹ️  Sin registros en landing staging — sin datos nuevos')
    notebookutils.notebook.exit('sin_datos_nuevos')


In [ ]:
# ============================================================
# Castear TODO a string (requisito Bronze TP)
# Agregar columnas de auditoría
# Carga incremental: append
# ============================================================

fecha_carga = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Castear todo a string y limpiar nombres de columnas
df_bronze = df_raw.select([
    F.col(c).cast(StringType())
     .alias(re.sub(r'[ ,;{}()\n\t=]', '_', c.strip()))
    for c in df_raw.columns
])

df_bronze = (
    df_bronze
    .withColumn('fecha_carga', F.lit(fecha_carga))
    .withColumn('origen_capa', F.lit('ADVWDB'))
)

# Incremental: append con deduplicación (idempotente)
df_bronze = df_bronze.dropDuplicates(['SalesOrderID', 'SalesOrderDetailID'])

(
    df_bronze.write
    .format('delta')
    .mode('append')
    .option('mergeSchema', 'true')
    .saveAsTable(tabla_bronze)
)

filas_escritas = df_bronze.count()
print(f'✅ {filas_escritas} filas → Bronze ({tabla_bronze})')

# Limpiar staging en landing
spark.sql(f'DROP TABLE IF EXISTS {tabla_staging_landing}')
print(f'🗑️  Staging landing eliminado: {tabla_staging_landing}')


In [ ]:
# ============================================================
# Verificar Bronze
# ============================================================

total = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabla_bronze}').collect()[0]['cnt']
print(f'✅ Total filas en Bronze ({tabla_bronze}): {total}')
display(spark.sql(f'SELECT * FROM {tabla_bronze} LIMIT 5'))
